# Train HR Hiring Agent with Hugging Face TRL & OpenEnv
This notebook provides a minimal training script using **Hugging Face TRL (PPO)** to train an LLM on the **Autonomous HR Hiring Agent** environment.

In [ ]:
!pip install trl transformers torch openenv

In [ ]:
import os
import json
import torch
from openenv import OpenEnv
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer

# To run this, you need the hr-hiring-env in your colab environment.
# We'll assume the files are uploaded or cloned via git.
import sys
sys.path.append('./hr-hiring-env') # ensure src.env can be imported if cloned

from src.env import HRHiringEnv

In [ ]:
# 1. Initialize the Environment
print("Initializing HR Hiring Environment...")
env = HRHiringEnv()

# 2. Setup Model and Tokenizer (using a small model for demonstration)
model_name = "Qwen/Qwen2.5-0.5B-Instruct" 
print(f"Loading Model: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load model with value head for PPO
model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


In [ ]:
# 3. Setup PPO Trainer
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=4,
    mini_batch_size=1,
    gradient_accumulation_steps=1,
    optimize_cuda_cache=True,
)

trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
)


In [ ]:
# 4. Minimal Training Loop
print("Starting RL Training Loop...")
num_episodes = 50

for episode in range(num_episodes):
    obs = env.reset()
    done = False
    
    episode_reward = 0
    step_count = 0
    
    # Store trajectory for PPO update
    queries = []
    responses = []
    rewards = []
    
    while not done:
        prompt = f"Observation: {json.dumps(obs)}\nAction (JSON):"
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.pretrained_model.device)
        
        with torch.no_grad():
            output_ids = model.generate(input_ids, max_new_tokens=150, do_sample=True, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
        
        response_tensor = output_ids[0][input_ids.shape[1]:]
        response_text = tokenizer.decode(response_tensor, skip_special_tokens=True)
        
        try:
            action_dict = json.loads(response_text)
        except Exception:
            action_dict = {"action_type": "declare_done", "memory_scratchpad": "Failed to parse JSON."}
            
        obs, reward, done, info = env.step(action_dict)
        episode_reward += reward
        step_count += 1
        
        queries.append(input_ids[0])
        responses.append(response_tensor)
        rewards.append(torch.tensor(reward, dtype=torch.float16))
    
    print(f"Episode {episode+1}/{num_episodes} - Steps: {step_count} - Total Reward: {episode_reward:.2f}")
    
    # 5. PPO Update Step
    if len(queries) > 0:
        trainer.step([queries[0]], [responses[0]], [rewards[-1]]) # Single batch for simplicity

print("Training Complete.")
